# Embedded C — Part 1: Bits, Registers and Hardware-Level Data

Embedded C is ordinary C used close to the hardware: microcontrollers like STM32, AVR (Arduino), ESP32, or PIC. The language is the same, but the priorities change:

| Desktop C | Embedded C |
|---|---|
| plenty of RAM, heap is normal | a few KB of RAM; heap often avoided |
| `int` size rarely matters | exact sizes matter: use `uint8_t`, `uint32_t` |
| talk to the OS | talk directly to **hardware registers** at fixed addresses |
| program exits | program never ends: `for (;;)` super loop |
| crashes show an error | crashes may silently lock up a device |

Colab has no microcontroller attached, so these notebooks **simulate** the hardware on the PC. The code patterns (bit masks, register structs, `volatile`, drivers) are exactly what you'd write for a real chip. At the end there's an optional cell that cross-compiles real bare-metal code for an ARM Cortex-M.

**Prerequisite:** the *C Language Complete Course* notebook, especially pointers, structs and the modular-programming part.

## Files created

```
/content/embedded_c/part1/
├── include/
│   ├── bitops.h        reusable bit-manipulation macros + helpers
│   └── mcu_sim.h       a simulated GPIO peripheral (register map)
└── src/
    ├── fixed_width.c   stdint.h types, overflow, integer promotion
    ├── bitops.c        implementation of bitops.h
    ├── bits_demo.c     using the macros on a control register
    ├── registers_demo.c memory-mapped registers via struct overlay
    ├── volatile_demo.c why volatile matters (look at the assembly)
    ├── struct_layout.c padding, packing, bit-fields
    ├── endian.c        byte order and portable byte packing
    └── blinky_bare.c   real STM32 bare-metal code (cross-compile only)
```

## 0. Setup

In [ ]:
!mkdir -p /content/embedded_c/part1/include /content/embedded_c/part1/src
!gcc --version | head -n 1

---
## 1. Fixed-width integer types (`<stdint.h>`)

On a desktop an `int` is 32 bits, but on an 8-bit AVR or a 16-bit MSP430 it's 16 bits. Registers and communication protocols need **exact** sizes, so embedded code uses:

| Type | Bits | Range |
|---|---|---|
| `uint8_t` / `int8_t` | 8 | 0..255 / −128..127 |
| `uint16_t` / `int16_t` | 16 | 0..65535 / −32768..32767 |
| `uint32_t` / `int32_t` | 32 | 0..4.29 billion / ±2.1 billion |
| `uint64_t` / `int64_t` | 64 | very large |

Three traps to understand:

1. **Unsigned overflow wraps around** (255 + 1 becomes 0 in a `uint8_t`). That's well defined. **Signed** overflow is undefined behaviour, so never rely on it.
2. **Integer promotion:** arithmetic on types smaller than `int` is done as `int` first, then converted when stored back.
3. **Widen before multiplying:** `adc * 3300` can overflow a 16-bit `int` on a small MCU, so cast to `uint32_t` first.

`<inttypes.h>` provides portable `printf` formats like `PRIu32`.

In [ ]:
%%writefile /content/embedded_c/part1/src/fixed_width.c
#include <stdio.h>
#include <stdint.h>
#include <inttypes.h>   // PRIu8, PRIu32 ... portable printf formats

int main(void)
{
    uint8_t  status  = 0xFF;          /* exactly 8 bits, 0..255  */
    int16_t  temp    = -40;           /* exactly 16 bits, signed */
    uint32_t counter = 4000000000u;   /* exactly 32 bits         */

    printf("uint8_t  %zu byte(s): %" PRIu8  "\n", sizeof status,  status);
    printf("int16_t  %zu byte(s): %" PRId16 "\n", sizeof temp,    temp);
    printf("uint32_t %zu byte(s): %" PRIu32 "\n\n", sizeof counter, counter);

    status++;                         /* unsigned overflow wraps (well defined) */
    printf("0xFF + 1 in a uint8_t = %" PRIu8 "\n", status);

    uint8_t a = 200, b = 100;
    uint8_t sum8 = (uint8_t)(a + b);  /* computed as int 300, then cut to 8 bits */
    int     sum  = a + b;             /* integer promotion: a and b become int */
    printf("200 + 100 -> uint8_t: %" PRIu8 ", int: %d\n\n", sum8, sum);

    uint16_t adc = 3000;                              /* 12-bit ADC reading (0..4095) */
    uint32_t mv  = (uint32_t)adc * 3300u / 4095u;     /* widen BEFORE multiplying */
    printf("ADC %" PRIu16 " -> %" PRIu32 " mV (3.3 V reference)\n", adc, mv);
    return 0;
}

In [ ]:
!cd /content/embedded_c/part1 && gcc -Wall -Wextra -std=c11 src/fixed_width.c -o fixed_width && ./fixed_width

---
## 2. Bit manipulation

Hardware is controlled one **bit** at a time: bit 5 of a port register might drive an LED, bits 1–3 of a timer register might select a mode. These four operations are the alphabet of embedded C:

| Goal | Code | Why it works |
|---|---|---|
| set bit n | `reg \|= (1u << n)` | OR with 1 forces it to 1 |
| clear bit n | `reg &= ~(1u << n)` | AND with 0 forces it to 0 |
| toggle bit n | `reg ^= (1u << n)` | XOR with 1 flips it |
| read bit n | `(reg >> n) & 1u` | shift it down, mask the rest |

For a **multi-bit field** (say 3 bits starting at bit 1), build a mask, clear the field, then OR in the new value. That's a **read-modify-write**, and it leaves the other bits untouched.

Use `1u` (unsigned), not `1`: shifting a signed `1` into the top bit is undefined behaviour.

We put these in a reusable header, `bitops.h`, the way a real project would.

In [ ]:
%%writefile /content/embedded_c/part1/include/bitops.h
#ifndef BITOPS_H
#define BITOPS_H

#include <stdint.h>

#define BIT(n)                 (1u << (n))
#define SET_BIT(reg, n)        ((reg) |=  BIT(n))
#define CLEAR_BIT(reg, n)      ((reg) &= ~BIT(n))
#define TOGGLE_BIT(reg, n)     ((reg) ^=  BIT(n))
#define READ_BIT(reg, n)       (((reg) >> (n)) & 1u)

/* Multi-bit fields: width = how many bits, pos = lowest bit of the field */
#define FIELD_MASK(width, pos)         (((1u << (width)) - 1u) << (pos))
#define FIELD_GET(reg, width, pos)     (((reg) & FIELD_MASK(width, pos)) >> (pos))
#define FIELD_SET(reg, width, pos, v)  ((reg) = ((reg) & ~FIELD_MASK(width, pos)) | \
                                         (((uint32_t)(v) << (pos)) & FIELD_MASK(width, pos)))

void print_bits8(uint8_t value);
void print_bits32(uint32_t value);
int  count_set_bits(uint32_t value);
int  is_power_of_two(uint32_t value);

#endif /* BITOPS_H */

In [ ]:
%%writefile /content/embedded_c/part1/src/bitops.c
#include <stdio.h>
#include "bitops.h"

void print_bits8(uint8_t value)
{
    for (int i = 7; i >= 0; i--) putchar(READ_BIT(value, i) ? '1' : '0');
}

void print_bits32(uint32_t value)
{
    for (int i = 31; i >= 0; i--) {
        putchar(READ_BIT(value, i) ? '1' : '0');
        if (i % 4 == 0 && i != 0) putchar('_');   /* group in nibbles for readability */
    }
}

int count_set_bits(uint32_t value)
{
    int count = 0;
    while (value) {
        value &= value - 1;   /* trick: clears the lowest set bit */
        count++;
    }
    return count;
}

int is_power_of_two(uint32_t value)
{
    return value != 0 && (value & (value - 1)) == 0;
}

In [ ]:
%%writefile /content/embedded_c/part1/src/bits_demo.c
#include <stdio.h>
#include "bitops.h"

int main(void)
{
    uint8_t port = 0x00;
    printf("start         : "); print_bits8(port); printf("\n");
    SET_BIT(port, 0);    printf("SET bit 0     : "); print_bits8(port); printf("\n");
    SET_BIT(port, 5);    printf("SET bit 5     : "); print_bits8(port); printf("\n");
    TOGGLE_BIT(port, 7); printf("TOGGLE bit 7  : "); print_bits8(port); printf("\n");
    CLEAR_BIT(port, 0);  printf("CLEAR bit 0   : "); print_bits8(port); printf("\n");
    printf("READ bit 5 = %u, bit 1 = %u\n\n", READ_BIT(port, 5), READ_BIT(port, 1));

    /* A made-up timer control register:
         bit  0      ENABLE
         bits 1..3   MODE       (3 bits)
         bits 4..7   PRESCALER  (4 bits)
         bits 8..15  PERIOD     (8 bits)  */
    uint32_t ctrl = 0;
    SET_BIT(ctrl, 0);
    FIELD_SET(ctrl, 3, 1, 5);      /* MODE = 5 */
    FIELD_SET(ctrl, 4, 4, 9);      /* PRESCALER = 9 */
    FIELD_SET(ctrl, 8, 8, 200);    /* PERIOD = 200 */
    printf("ctrl = 0x%08X\n       ", (unsigned)ctrl);
    print_bits32(ctrl);
    printf("\nENABLE=%u MODE=%u PRESCALER=%u PERIOD=%u\n\n",
           READ_BIT(ctrl, 0), (unsigned)FIELD_GET(ctrl, 3, 1),
           (unsigned)FIELD_GET(ctrl, 4, 4), (unsigned)FIELD_GET(ctrl, 8, 8));

    FIELD_SET(ctrl, 3, 1, 2);      /* change only MODE; everything else is kept */
    printf("after MODE=2: ctrl = 0x%08X, PERIOD still %u\n\n",
           (unsigned)ctrl, (unsigned)FIELD_GET(ctrl, 8, 8));

    printf("set bits in 0xF0F0 = %d\n", count_set_bits(0xF0F0));
    printf("is 64 a power of two? %d   is 96? %d\n", is_power_of_two(64), is_power_of_two(96));
    return 0;
}

In [ ]:
!cd /content/embedded_c/part1 && gcc -Wall -Wextra -std=c11 -Iinclude src/bitops.c src/bits_demo.c -o bits_demo && ./bits_demo

---
## 3. Memory-mapped registers

On a microcontroller, peripherals (GPIO, UART, timers, ADC) are controlled through **registers** that appear at fixed memory addresses. Writing to the right address turns on an LED; reading another tells you a button's state. In C, you reach an address through a pointer:

```c
#define GPIOA_ODR (*(volatile uint32_t *)0x40020014u)   // STM32F4 output data register
GPIOA_ODR |= (1u << 5);                                 // drive pin PA5 high
```

Vendor headers (like STM32's CMSIS files) go one step further: they describe a whole peripheral as a **struct** whose members line up with the register offsets, then point the struct at the base address. Every member is `volatile` (see section 4).

Since Colab has no hardware, `mcu_sim.h` points the struct at an ordinary array instead of address `0x40020000`. The driver code is written exactly as it would be on a chip, and a small function plays the part of the silicon.

**BSRR (bit set/reset register):** writing 1 to bit *n* sets pin *n*, writing 1 to bit *n+16* clears it. That's a single write with no read-modify-write, so an interrupt can't corrupt it halfway. Real STM32 code uses it heavily.

> The register map here is simplified to four registers. A real STM32 GPIO block has more (OTYPER, OSPEEDR, PUPDR…), and the offsets in the reference manual differ.

In [ ]:
%%writefile /content/embedded_c/part1/include/mcu_sim.h
#ifndef MCU_SIM_H
#define MCU_SIM_H

#include <stdint.h>

/* On a real chip this block of registers lives at a fixed address,
   for example  #define GPIOA_BASE 0x40020000u.
   Here we fake that memory with an array so the code can run on a PC. */

typedef struct {
    volatile uint32_t MODER;   /* offset 0x00: pin mode, 2 bits per pin */
    volatile uint32_t IDR;     /* offset 0x04: input data (pin levels)  */
    volatile uint32_t ODR;     /* offset 0x08: output data              */
    volatile uint32_t BSRR;    /* offset 0x0C: atomic bit set / reset   */
} GPIO_TypeDef;

extern uint32_t fake_peripheral_memory[16];

#define GPIOA ((GPIO_TypeDef *)&fake_peripheral_memory[0])
#define GPIOB ((GPIO_TypeDef *)&fake_peripheral_memory[4])

#define GPIO_MODE_INPUT   0u
#define GPIO_MODE_OUTPUT  1u

#endif /* MCU_SIM_H */

In [ ]:
%%writefile /content/embedded_c/part1/src/registers_demo.c
#include <stdio.h>
#include <stddef.h>     // offsetof
#include "mcu_sim.h"

uint32_t fake_peripheral_memory[16];   /* stands in for the chip's peripheral address space */

/* Pretend to be the silicon. Real hardware does this by itself. */
static void sim_hardware_update(GPIO_TypeDef *port)
{
    uint32_t bsrr = port->BSRR;
    port->ODR |=  (bsrr & 0xFFFFu);    /* lower 16 bits: set pins   */
    port->ODR &= ~(bsrr >> 16);        /* upper 16 bits: reset pins */
    port->BSRR = 0;                    /* BSRR always reads back as 0 */
}

static void gpio_set_mode(GPIO_TypeDef *port, unsigned pin, uint32_t mode)
{
    port->MODER &= ~(3u << (pin * 2));     /* clear the pin's 2 mode bits */
    port->MODER |=  (mode << (pin * 2));   /* write the new mode          */
}

static const char *led_state(void)
{
    return (GPIOA->ODR & (1u << 5)) ? "ON" : "OFF";
}

int main(void)
{
    printf("Register offsets: MODER=0x%02zX IDR=0x%02zX ODR=0x%02zX BSRR=0x%02zX\n\n",
           offsetof(GPIO_TypeDef, MODER), offsetof(GPIO_TypeDef, IDR),
           offsetof(GPIO_TypeDef, ODR),   offsetof(GPIO_TypeDef, BSRR));

    gpio_set_mode(GPIOA, 5, GPIO_MODE_OUTPUT);   /* PA5 drives the LED on many boards */
    printf("MODER = 0x%08X  (pin 5 set to output)\n", (unsigned)GPIOA->MODER);

    GPIOA->ODR |= (1u << 5);                     /* read-modify-write: LED on */
    printf("ODR after |=           : 0x%08X  LED %s\n", (unsigned)GPIOA->ODR, led_state());

    GPIOA->BSRR = (1u << (5 + 16));              /* atomic reset of pin 5 */
    sim_hardware_update(GPIOA);
    printf("ODR after BSRR reset 5 : 0x%08X  LED %s\n", (unsigned)GPIOA->ODR, led_state());

    GPIOA->BSRR = (1u << 5) | (1u << 6);         /* set pins 5 and 6 in ONE write */
    sim_hardware_update(GPIOA);
    printf("ODR after BSRR set 5,6 : 0x%08X  LED %s\n\n", (unsigned)GPIOA->ODR, led_state());

    fake_peripheral_memory[4 + 1] = 0x0001;      /* "hardware" pulls PB0 high */
    if (GPIOB->IDR & 1u) printf("Button on PB0 is pressed\n");
    return 0;
}

In [ ]:
!cd /content/embedded_c/part1 && gcc -Wall -Wextra -std=c11 -Iinclude src/registers_demo.c -o registers_demo && ./registers_demo

---
## 4. `volatile`: telling the compiler "this can change behind your back"

The optimiser assumes memory only changes when **your code** changes it. A hardware register, or a variable modified inside an interrupt, breaks that assumption. Without `volatile`, the compiler may read the value **once** and never again, turning "wait until the flag is set" into an infinite loop.

Use `volatile` for:

- hardware registers,
- variables shared between an interrupt handler and the main code,
- variables shared with DMA or another processor core.

Here are two identical loops compiled with optimisation (`-O2`). Compare the generated assembly:

In [ ]:
%%writefile /content/embedded_c/part1/src/volatile_demo.c
#include <stdint.h>

uint32_t          plain_flag;   /* compiler assumes nobody else changes this */
volatile uint32_t hw_flag;      /* compiler must re-read this every time      */

void wait_plain(void)
{
    while (plain_flag == 0) { }   /* wait for "hardware" to set the flag */
}

void wait_volatile(void)
{
    while (hw_flag == 0) { }
}

In [ ]:
!cd /content/embedded_c/part1 && gcc -O2 -S -fno-asynchronous-unwind-tables -fcf-protection=none -o - src/volatile_demo.c | sed -n '/^wait_plain:/,/ret$/p; /^wait_volatile:/,/ret$/p' | grep -vE '^\s+\.'

**How to read it:** in `wait_plain`, the compiler loads `plain_flag` **once** (`movl`). If it was 0, execution falls into `.L3: jmp .L3`, a loop that jumps to itself forever and never reads memory again (label numbers may differ slightly on your GCC version). In `wait_volatile`, the `movl` that reads `hw_flag` sits **inside** the loop (`.L6` … `je .L6`), so every pass checks the real value.

`volatile` does **not** make an operation atomic, and it doesn't protect a multi-step read-modify-write from being interrupted. For that you briefly disable interrupts or use atomic instructions (Part 2 shows the interrupt side).

---
## 5. Struct layout: padding, packing and bit-fields

The compiler inserts invisible **padding** bytes so each member sits at an address its CPU can read efficiently (a `uint32_t` at a multiple of 4). This matters when RAM is tight, and it really matters when a struct must match a fixed binary format, like a sensor packet or a file header.

- **Reorder** members from largest to smallest to cut padding for free.
- `__attribute__((packed))` (GCC) removes padding, but unaligned access is slower and **faults** on some cores (Cortex-M0, for example). Prefer copying bytes in and out explicitly.
- **Bit-fields** (`unsigned mode : 3;`) look perfect for registers, but bit order and layout are compiler-defined, so they aren't portable. Most professional driver code uses masks and shifts (section 2) instead.

In [ ]:
%%writefile /content/embedded_c/part1/src/struct_layout.c
#include <stdio.h>
#include <stdint.h>
#include <stddef.h>

typedef struct {
    uint8_t  id;        /* 1 byte, then 3 bytes of padding */
    uint32_t value;     /* 4 bytes, must start at a multiple of 4 */
    uint8_t  flags;     /* 1 byte, then 3 more bytes of padding at the end */
} Padded;

typedef struct {        /* same members, largest first */
    uint32_t value;
    uint8_t  id;
    uint8_t  flags;
} Reordered;

typedef struct __attribute__((packed)) {
    uint8_t  id;
    uint32_t value;
    uint8_t  flags;
} Packed;

typedef struct {
    unsigned enable    : 1;
    unsigned mode      : 3;
    unsigned prescaler : 4;
} CtrlBits;

int main(void)
{
    printf("Padded    : %2zu bytes  (id@%zu value@%zu flags@%zu)\n", sizeof(Padded),
           offsetof(Padded, id), offsetof(Padded, value), offsetof(Padded, flags));
    printf("Reordered : %2zu bytes\n", sizeof(Reordered));
    printf("Packed    : %2zu bytes  (value@%zu: unaligned, slow or faulting on some CPUs)\n",
           sizeof(Packed), offsetof(Packed, value));

    CtrlBits c = { .enable = 1, .mode = 5, .prescaler = 9 };
    printf("\nbit-field struct: %zu bytes, enable=%u mode=%u prescaler=%u\n",
           sizeof c, (unsigned)c.enable, (unsigned)c.mode, (unsigned)c.prescaler);

    unsigned too_big = 9;          /* 1001 in binary needs 4 bits */
    c.mode = too_big;              /* mode only has 3 bits */
    printf("after mode = 9: mode = %u (silently truncated to 3 bits)\n", (unsigned)c.mode);
    return 0;
}

In [ ]:
!cd /content/embedded_c/part1 && gcc -Wall -Wextra -std=c11 src/struct_layout.c -o struct_layout && ./struct_layout

---
## 6. Endianness

A multi-byte value like `0x12345678` can be stored lowest byte first (**little-endian**: x86, almost all ARM Cortex-M) or highest byte first (**big-endian**: many network protocols and some sensors).

Never `memcpy` raw bytes from a packet straight into an integer and hope. Build the value **explicitly from bytes** in the order the datasheet or protocol specifies. That code works on any CPU.

In [ ]:
%%writefile /content/embedded_c/part1/src/endian.c
#include <stdio.h>
#include <stdint.h>

static int is_little_endian(void)
{
    uint16_t x = 0x0001;
    return *(uint8_t *)&x == 0x01;      /* is the low byte stored first? */
}

static uint32_t swap32(uint32_t v)
{
    return ((v & 0x000000FFu) << 24) | ((v & 0x0000FF00u) << 8) |
           ((v & 0x00FF0000u) >> 8)  | ((v & 0xFF000000u) >> 24);
}

/* Portable: build the value from bytes in a known order */
static uint16_t read_be16(const uint8_t *p) { return (uint16_t)((p[0] << 8) | p[1]); }
static uint16_t read_le16(const uint8_t *p) { return (uint16_t)((p[1] << 8) | p[0]); }

static void write_be16(uint8_t *p, uint16_t v)
{
    p[0] = (uint8_t)(v >> 8);
    p[1] = (uint8_t)(v & 0xFF);
}

int main(void)
{
    printf("This CPU is %s-endian\n\n", is_little_endian() ? "little" : "big");

    uint32_t v = 0x12345678;
    uint8_t *b = (uint8_t *)&v;
    printf("0x%08X sits in memory as: %02X %02X %02X %02X\n", (unsigned)v, b[0], b[1], b[2], b[3]);
    printf("swap32 -> 0x%08X\n\n", (unsigned)swap32(v));

    uint8_t packet[2] = { 0x01, 0xF4 };    /* a sensor sends 500 as big-endian bytes */
    printf("bytes 01 F4 read as big-endian = %u, as little-endian = %u\n",
           read_be16(packet), read_le16(packet));

    uint8_t out[2];
    write_be16(out, 1234);
    printf("1234 written big-endian -> %02X %02X\n", out[0], out[1]);
    return 0;
}

In [ ]:
!cd /content/embedded_c/part1 && gcc -Wall -Wextra -std=c11 src/endian.c -o endian && ./endian

---
## 7. (Optional) Cross-compiling real bare-metal code for ARM Cortex-M

Everything above ran on Colab's x86 CPU. Firmware for a microcontroller is built with a **cross-compiler**, a GCC that runs on your PC but emits machine code for another CPU. For ARM Cortex-M chips (STM32, nRF52, RP2040, …) that's `arm-none-eabi-gcc`.

The file below is the classic "blinky" for an STM32F4 board, using real register addresses from the reference manual. It **can't run in Colab** (it would write to addresses that don't exist on a PC), but we can compile it and look at the ARM instructions it produces.

The install takes a minute or two. On a real project you'd also need a startup file (vector table + reset handler) and a linker script; vendor tools like STM32CubeIDE generate those for you.

In [ ]:
%%writefile /content/embedded_c/part1/src/blinky_bare.c
/* Bare-metal LED blink for an STM32F4 (e.g. Nucleo-F401RE, LED on PA5).
   No libraries, no HAL: just registers. */
#include <stdint.h>

#define RCC_AHB1ENR  (*(volatile uint32_t *)0x40023830u)   /* peripheral clock enable */
#define GPIOA_MODER  (*(volatile uint32_t *)0x40020000u)   /* GPIOA pin modes         */
#define GPIOA_ODR    (*(volatile uint32_t *)0x40020014u)   /* GPIOA output data       */

#define LED_PIN 5u

static void delay(volatile uint32_t count)   /* volatile: stop the optimiser deleting it */
{
    while (count--) { }
}

int main(void)
{
    RCC_AHB1ENR |= (1u << 0);                    /* 1. turn on the clock for GPIOA */

    GPIOA_MODER &= ~(3u << (LED_PIN * 2));       /* 2. make PA5 an output */
    GPIOA_MODER |=  (1u << (LED_PIN * 2));

    for (;;) {                                   /* 3. super loop: never returns */
        GPIOA_ODR ^= (1u << LED_PIN);            /*    toggle the LED */
        delay(500000);
    }
}

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y gcc-arm-none-eabi > /dev/null && arm-none-eabi-gcc --version | head -n 1

In [ ]:
!cd /content/embedded_c/part1 && arm-none-eabi-gcc -mcpu=cortex-m4 -mthumb -O2 -Wall -c src/blinky_bare.c -o blinky_bare.o && echo "=== size (bytes) ===" && arm-none-eabi-size blinky_bare.o && echo && echo "=== ARM Thumb assembly ===" && arm-none-eabi-objdump -d blinky_bare.o

The whole program is a few dozen bytes of ARM Thumb code. Look for the `ldr`/`str` instructions: those are the reads and writes to the register addresses, and `eor` is the XOR that toggles the LED.

---
# Exercises

1. Write `uint8_t reverse_bits(uint8_t b)` (e.g. `0b00000001` → `0b10000000`) and test it with `print_bits8`.
2. Add a `WRITE_BIT(reg, n, value)` macro to `bitops.h` that sets or clears bit `n` depending on `value`.
3. Extend `mcu_sim.h` with a simulated UART block (`SR`, `DR`, `BRR`) and write `uart_putc()` that waits for a "TX empty" bit in `SR`. Add the "hardware" side so it sets that bit.
4. A temperature sensor sends a **signed** 16-bit big-endian value in 0.01 °C steps. Decode the bytes `FF 38` (should be −2.00 °C).
5. Reorder this struct to make it as small as possible: `{ uint8_t a; uint64_t b; uint16_t c; uint8_t d; uint32_t e; }`. Check with `sizeof`.

**Next:** *Embedded C — Part 2: Drivers, State Machines and Interrupts.*